# Unit 3 — Wavelet Transform Analysis

**Syllabus coverage:** Unit 3 — Wavelet Transforms

**Experiments covered:** #5 (DWT Decomposition), #6 (Regime Change Detection)

We decompose the entropy signal using discrete and continuous wavelet transforms
to analyse multi-scale structure and detect regime changes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pywt
from pathlib import Path

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('seaborn-whitegrid')

REPO_ROOT = Path('.').resolve().parent
DATA_DIR = REPO_ROOT / 'data' / 'processed'
FIG_DIR = REPO_ROOT / 'outputs' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
SAVEKW = dict(dpi=150, bbox_inches='tight')

## Section 1 — DWT Decomposition (Experiment #5)

In [ ]:
df = pd.read_csv(DATA_DIR / 'entropy_signal.csv', parse_dates=['meet_date'])
H = df['entropy_H'].dropna().values
print(f'Signal length: {len(H)}')

# DWT decomposition with db4 wavelet, 6 levels
wavelet = 'db4'
max_level = 6
coeffs = pywt.wavedec(H, wavelet, level=max_level)
# coeffs[0] = approximation (cA6), coeffs[1:] = details (cD6, cD5, ..., cD1)

print(f'Wavelet: {wavelet}, Levels: {max_level}')
for i, c in enumerate(coeffs):
    label = 'cA6' if i == 0 else f'cD{max_level - i + 1}'
    print(f'  {label}: {len(c)} coefficients')

In [ ]:
# Plot approximation and detail coefficients
fig, axes = plt.subplots(max_level + 2, 1, figsize=(16, 3 * (max_level + 2)), sharex=False)

# Original signal
axes[0].plot(H, lw=0.3, color='steelblue')
axes[0].set_title('Original Entropy Signal')
axes[0].set_ylabel('H (bits)')

# Approximation coefficients
axes[1].plot(coeffs[0], lw=0.8, color='navy')
axes[1].set_title(f'Approximation Coefficients (cA{max_level}) — Trend')
axes[1].set_ylabel('cA6')

# Detail coefficients
for level_idx in range(1, max_level + 1):
    ax = axes[level_idx + 1]
    detail_level = max_level - level_idx + 1
    ax.plot(coeffs[level_idx], lw=0.5, color='coral')
    ax.set_title(f'Detail Coefficients (cD{detail_level})')
    ax.set_ylabel(f'cD{detail_level}')

axes[-1].set_xlabel('Coefficient index')
fig.tight_layout()
fig.savefig(FIG_DIR / 'unit3_dwt_decomposition.png', **SAVEKW)
plt.show()

In [ ]:
# Perfect reconstruction verification
reconstructed = pywt.waverec(coeffs, wavelet)
# Trim to original length (waverec may add 1 sample)
reconstructed = reconstructed[:len(H)]
recon_error = np.max(np.abs(H - reconstructed))
print(f'Max reconstruction error: {recon_error:.2e}')
print(f'Perfect reconstruction: {recon_error < 1e-10}')

## Section 2 — Multi-scale Entropy Analysis

In [ ]:
# Energy distribution across scales
energies = []
labels = []
for i, c in enumerate(coeffs):
    e = np.sum(c ** 2)
    label = f'cA{max_level}' if i == 0 else f'cD{max_level - i + 1}'
    energies.append(e)
    labels.append(label)
    print(f'{label}: energy = {e:.2f}, fraction = {e / sum(e2 ** 2 for c2 in coeffs for e2 in c2) * len(c):.4f}')

total_energy = sum(energies)
frac = [e / total_energy for e in energies]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(labels, frac, color='teal', edgecolor='white')
ax.set_xlabel('Wavelet Component')
ax.set_ylabel('Energy Fraction')
ax.set_title('Energy Distribution Across DWT Scales')
for i, (f, e) in enumerate(zip(frac, energies)):
    ax.text(i, f + 0.005, f'{f:.2%}', ha='center', fontsize=9)
fig.savefig(FIG_DIR / 'unit3_energy_distribution.png', **SAVEKW)
plt.show()

In [ ]:
# Energy distribution stratified by entropy bin
fig, ax = plt.subplots(figsize=(12, 5))
width = 0.25
x = np.arange(max_level + 1)

for idx, bin_label in enumerate(['low', 'medium', 'high']):
    mask = df['entropy_bin'] == bin_label
    H_sub = df.loc[mask, 'entropy_H'].dropna().values
    if len(H_sub) < 2 ** max_level:
        continue
    c_sub = pywt.wavedec(H_sub, wavelet, level=max_level)
    e_sub = [np.sum(c ** 2) for c in c_sub]
    e_total = sum(e_sub)
    frac_sub = [e / e_total for e in e_sub]
    ax.bar(x + idx * width, frac_sub, width, label=bin_label, alpha=0.8)

ax.set_xticks(x + width)
ax.set_xticklabels(labels)
ax.set_xlabel('Wavelet Component')
ax.set_ylabel('Energy Fraction')
ax.set_title('Energy Distribution by Entropy Bin')
ax.legend()
fig.savefig(FIG_DIR / 'unit3_energy_by_bin.png', **SAVEKW)
plt.show()

## Section 3 — Regime Change Detection (Experiment #6)

In [ ]:
# CWT with Mexican hat wavelet
scales = np.arange(1, 128)
cwt_coeffs, cwt_freqs = pywt.cwt(H, scales, 'mexh')

# Scalogram
fig, ax = plt.subplots(figsize=(16, 6))
im = ax.pcolormesh(np.arange(len(H)), scales, np.abs(cwt_coeffs),
                   cmap='magma', shading='auto')
ax.set_xlabel('Race Sequence Index')
ax.set_ylabel('CWT Scale')
ax.set_title('Continuous Wavelet Transform Scalogram (Mexican Hat)')
fig.colorbar(im, ax=ax, label='|CWT Coefficient|')
fig.savefig(FIG_DIR / 'unit3_scalogram.png', **SAVEKW)
plt.show()

In [ ]:
# Detect change points from aggregated CWT power
from scipy.signal import find_peaks

power = np.sum(np.abs(cwt_coeffs), axis=0)
peaks, props = find_peaks(power, distance=50, prominence=np.std(power))

# Top 10 by prominence
if len(peaks) > 10:
    top_idx = np.argsort(props['prominences'])[-10:]
    top_peaks = sorted(peaks[top_idx])
else:
    top_peaks = sorted(peaks)

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
axes[0].plot(H, lw=0.3, alpha=0.5, color='steelblue')
rolling = pd.Series(H).rolling(50, center=True).mean()
axes[0].plot(rolling, lw=1.5, color='navy')
for cp in top_peaks:
    axes[0].axvline(cp, color='red', alpha=0.6, lw=1)
axes[0].set_ylabel('Entropy H (bits)')
axes[0].set_title('Entropy Signal with Top 10 Wavelet Change Points')

axes[1].plot(power, lw=0.5, color='purple')
for cp in top_peaks:
    axes[1].axvline(cp, color='red', alpha=0.6, lw=1)
axes[1].set_ylabel('Aggregated CWT Power')
axes[1].set_xlabel('Race Index')

fig.tight_layout()
fig.savefig(FIG_DIR / 'unit3_changepoints.png', **SAVEKW)
plt.show()

# List change points with dates
print('Top 10 Change Points:')
for cp in top_peaks:
    row = df.iloc[cp]
    print(f'  idx={cp:>5d}, seq_id={int(row["race_seq_id"]):>5d}, '
          f'date={pd.Timestamp(row["meet_date"]).strftime("%Y-%m-%d")}, '
          f'venue={row["venue"]}, H={row["entropy_H"]:.3f}')

## Summary

- DWT decomposition at 6 levels separates race-to-race noise (level 1) from seasonal trends (levels 5–6).
- Energy is concentrated in the approximation and lower detail levels, indicating strong low-frequency structure.
- CWT scalogram with Mexican hat wavelet identifies regime change points.
- Change points can be cross-referenced with real-world events (e.g. COVID lockdowns, seasonal breaks).